# Exploration scratchpad

A workbench for inspecting query results and iterating on Altair encodings.
Not a deliverable: when the charts settle, `quarto convert explore.ipynb`
turns this into a `.qmd` to write the narrative around.

Start it with:

```
uv run jupyter lab
```

**Restart-and-run-all before trusting anything here.** Out-of-order execution
is the classic way a notebook ends up depending on a cell you deleted.

In [1]:
import sys
from pathlib import Path

# scripts/ is not an installed package, so put the repo root on sys.path to let
# `from scripts.queries import ...` resolve. Path.cwd() is notebooks/ when Jupyter
# is launched from the repo root, so the root is its parent.
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import altair as alt
import pandas as pd

from scripts.queries import ALTAIR_MAX_ROWS, ANALYSIS_QUERIES, run_query, run_script

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 1. What shape is every query result?

Row count decides chart type. Anything over `ALTAIR_MAX_ROWS` cannot be plotted
row-per-record and needs aggregating in SQL first.

In [2]:
results = {name: run_query(name) for name in ANALYSIS_QUERIES}

pd.DataFrame(
    [
        {
            "query": name,
            "rows": len(df),
            "cols": len(df.columns),
            "over_altair_limit": len(df) > ALTAIR_MAX_ROWS,
        }
        for name, df in results.items()
    ]
)

,query,rows,cols,over_altair_limit
0,most_helpful_reviews,4315,9,False
1,product_summary,111,5,False
2,product_trajectory,3044,8,False
3,rating_vs_reviewer_avg,187,4,False
4,reviewer_summary,75,3,False


## 2. Inspect one result

Check dtypes here. SQLite is loose about types, so a `CASE` branch that returns
a string somewhere will show up as `object` where you expected `float64`.

In [3]:
df = results["product_summary"]
print(df.dtypes)
df.head()

stars_bin        float64
product_count      int64
total_reviews      int64
size_bucket          str
bucket_order       int64
dtype: object


,stars_bin,product_count,total_reviews,size_bucket,bucket_order
0,1.2,1,5,5-9,5
1,1.4,1,5,5-9,5
2,1.8,4,23,5-9,5
3,2.0,14,72,5-9,5
4,2.2,26,145,5-9,5


## 3. The integrity checks

`data_profiling.sql` holds three independent statements, so it needs
`run_script()`. `run_query()` would silently return only the first.

In [4]:
for i, check in enumerate(run_script("data_profiling"), start=1):
    print(f"--- statement {i} ---")
    print(check.to_string(index=False))
    print()

--- statement 1 ---
 null_brand  empty_str_brand  whitespace_brand  total_products
       5068                0                 0           10429

--- statement 2 ---
 reviews_no_product_row  reviews_no_title  distinct_asins_no_title  total_reviews
                      0              2841                      237         194439

--- statement 3 ---
 products_with_dupes
                   0



## 4. Chart theme

One registered Altair theme so the chart cells stay about encodings, not
styling. Colors are the validated reference palette from the dataviz method:
a sequential blue ramp for magnitude, one accent + gray for emphasis, and a
red↔blue diverging set (gray midpoint) for polarity.


In [5]:
# Palette (light mode). Sequential = one hue light->dark; diverging = two hue
# poles around a neutral gray midpoint; emphasis = one accent hue + gray.
ACCENT = "#2a78d6"       # blue accent
DEEMPHASIS = "#c3c2b7"   # gray for context lines in emphasis charts
SEQ_RAMP = [             # sequential blue ramp, light -> dark
    "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b",
]
DIVERGING_5 = [          # stars 1..5: red pole, neutral gray, blue pole
    "#c73433", "#e88a89", "#a7a69f", "#86b6ef", "#1c5cab",
]
SURFACE = "#fcfcfb"

# Vega expression for the 0.1-binned ordinal axes: labelling all 41 bins would
# collide, so keep only integer and .5 labels. datum.label is the tick's text.
SPARSE_DECIMAL_LABELS = (
    "test(/^[0-9]+$/, datum.label) || test(/[.]5$/, datum.label) ? datum.label : ''"
)


# alt.theme.register installs this as the default config for every chart in
# the session (enable=True activates it immediately).
@alt.theme.register("review_theme", enable=True)
def review_theme() -> alt.theme.ThemeConfig:
    return {
        "config": {
            "background": SURFACE,
            "font": 'system-ui, -apple-system, "Segoe UI", sans-serif',
            "view": {"stroke": None},
            "axis": {
                "labelColor": "#898781",
                "titleColor": "#52514e",
                "gridColor": "#e1e0d9",
                "domainColor": "#c3c2b7",
                "tickColor": "#c3c2b7",
                "titleFontWeight": "normal",
            },
            "legend": {"labelColor": "#52514e", "titleColor": "#52514e"},
            "title": {
                "color": "#0b0b0b",
                "fontSize": 14,
                "fontWeight": "bold",
                "anchor": "start",
                "subtitleColor": "#52514e",
                "subtitleFontSize": 12,
            },
        }
    }

## 5. rating_vs_reviewer_avg — heatmap

The query's grain is already a 2D histogram (stars given × reviewer's binned
lifetime average), and the job is magnitude over a grid, so the form is a
heatmap with a sequential ramp. Counts span 1 to ~14,000, so the color scale
is log — linear would flatten everything but the (5, 5.0) cell.


In [6]:
alt.Chart(results["rating_vs_reviewer_avg"]).mark_rect(
    # a hairline gap between cells, in the surface color, so fills never touch
    stroke=SURFACE,
    strokeWidth=2,
).encode(
    x=alt.X(
        "reviewer_avg_bin:O",
        title="Reviewer's lifetime average (binned to 0.1)",
        axis=alt.Axis(labelAngle=0, labelExpr=SPARSE_DECIMAL_LABELS),
    ),
    y=alt.Y("stars:O", sort="descending", title="Stars given"),
    color=alt.Color(
        "review_count:Q",
        scale=alt.Scale(type="log", range=SEQ_RAMP),
        title="Reviews (log)",
    ),
    tooltip=[
        alt.Tooltip("stars:O", title="Stars given"),
        alt.Tooltip("reviewer_avg_bin:O", title="Reviewer avg"),
        alt.Tooltip("review_count:Q", title="Reviews", format=","),
        alt.Tooltip("above_reviewer_avg:Q", title="Above own avg", format="+.1f"),
    ],
).properties(
    width=620,
    height=180,
    title=alt.Title(
        "Ratings vs. the reviewer's own average",
        subtitle="Each cell counts reviews; the diagonal is people rating at their usual level.",
    ),
)

alt.Chart(...)

You may notice that there are stripes of points with fewer reviews where the reviewer's lifetime average is at x.9. This is because a reviewer's average can only be x.9 if that reviewer has >= 10 reviews, which the majority of reviewers do not. I chose to keep the binning at 0.1 despite this, to not lose the more interesting aspects of this heatmap.

## 6. product_trajectory — emphasis multi-line

30 series is far past the point where distinct colors work (the categorical
ceiling is ~8), so this is the *emphasis* form instead: every trajectory in a
recessive gray, and hovering paints one line in the accent hue. Identity comes
from the hover tooltip, not a 30-entry legend.

The second, invisible 10px-wide line layer is the hover hit-target — a 1.2px
line is nearly impossible to pointerover directly.

The query hides each product's weeks before its 10th review (the burn-in
filter in `product_trajectory.sql`): a running average over a handful of
reviews is mostly noise, and those early segments dominated the chart while
saying nothing.


In [7]:
hover = alt.selection_point(
    fields=["asin"], on="pointerover", clear="pointerout", empty=False
)

trajectory_base = alt.Chart(results["product_trajectory"]).encode(
    x=alt.X("week:T", title=None, axis=alt.Axis(format="%Y", tickCount="year")),
    y=alt.Y("running_avg:Q", scale=alt.Scale(zero=False), title="Running average stars"),
    # detail splits the data into one line per product without assigning colors
    detail="asin:N",
)

trajectory_lines = trajectory_base.mark_line().encode(
    color=alt.condition(hover, alt.value(ACCENT), alt.value(DEEMPHASIS)),
    strokeWidth=alt.condition(hover, alt.value(2.5), alt.value(1.2)),
    opacity=alt.condition(hover, alt.value(1.0), alt.value(0.75)),
)

trajectory_hit = trajectory_base.mark_line(strokeWidth=10, opacity=0.001).encode(
    tooltip=[
        alt.Tooltip("title:N", title="Product"),
        alt.Tooltip("week:T", title="Week of"),
        alt.Tooltip("running_avg:Q", title="Running avg", format=".2f"),
        alt.Tooltip("n_reviews:Q", title="Reviews that week"),
        alt.Tooltip("delta:Q", title="Change vs prev week", format="+.3f"),
    ]
).add_params(hover)

(trajectory_lines + trajectory_hit).properties(
    width=620,
    height=300,
    title=alt.Title(
        "Rating trajectories of the 30 most-reviewed products",
        subtitle="Hover a line to identify the product. Weekly running average, shown from each product's 10th review onward.",
    ),
)

alt.LayerChart(...)

## 7. reviewer_summary — two stacked charts

Two measures on very different scales (reviewer counts up to ~12k, average
ratings between 3.9 and 5.0). Never a dual-axis chart — instead two charts
sharing the x-axis via `vconcat` + `resolve_scale`. The activity chart needs a
log y because the distribution is so heavy-tailed.


In [8]:
x_reviews = alt.X(
    "review_count:Q",
    # nice=False stops the log scale rounding its floor down to 1;
    # the data starts at the 5-core minimum of 5
    scale=alt.Scale(type="log", nice=False),
    title="Reviews written (log)",
    # explicit tick values: a log axis otherwise grids every minor step, and
    # powers of 10 alone would leave just 1/10/100 across this 1-149 range
    axis=alt.Axis(values=[5, 10, 20, 50, 100, 150]),
)

activity = alt.Chart(results["reviewer_summary"]).mark_circle(
    size=45, color=ACCENT, opacity=0.85
).encode(
    x=x_reviews,
    y=alt.Y(
        "reviewer_count:Q",
        scale=alt.Scale(type="log"),
        title="Reviewers (log)",
        # a log axis grids every minor tick by default (ruled-paper effect);
        # tickCount thins it, "~s" abbreviates 10000 -> 10k
        axis=alt.Axis(tickCount=4, format="~s"),
    ),
    tooltip=[
        alt.Tooltip("review_count:Q", title="Reviews written"),
        alt.Tooltip("reviewer_count:Q", title="Reviewers", format=","),
    ],
).properties(
    width=620,
    height=180,
    title=alt.Title(
        "How active are reviewers?",
        subtitle="Every reviewer has at least 5 reviews — the 5-core subset guarantee.",
    ),
)

leniency = alt.Chart(results["reviewer_summary"]).mark_circle(
    color=ACCENT, opacity=0.85
).encode(
    x=x_reviews,
    y=alt.Y(
        "avg_reviewer_rating:Q",
        scale=alt.Scale(zero=False),
        title="Avg rating given",
    ),
    # size shows how many reviewers each point summarizes, so the eye can
    # discount the noisy singleton points on the right
    size=alt.Size(
        "reviewer_count:Q",
        scale=alt.Scale(type="log", range=[20, 300]),
        title="Reviewers",
    ),
    tooltip=[
        alt.Tooltip("review_count:Q", title="Reviews written"),
        alt.Tooltip("avg_reviewer_rating:Q", title="Avg rating", format=".2f"),
        alt.Tooltip("reviewer_count:Q", title="Reviewers", format=","),
    ],
).properties(
    width=620,
    height=180,
    title=alt.Title(
        "Do prolific reviewers rate differently?",
        subtitle="Point size = how many reviewers sit at that activity level (log).",
    ),
)

alt.vconcat(activity, leniency).resolve_scale(x="shared")

alt.VConcatChart(...)

## 8. product_summary — heatmap

Same form logic as §5: the grain is a grid (avg-stars bin × review-volume
bucket) and the job is magnitude, so heatmap + sequential ramp + log color.
The bucket order comes from an explicit sort list — alphabetical would put
"100+" between "10-29" and "30-99".


In [9]:
SIZE_BUCKETS = ["5-9", "10-29", "30-99", "100+"]

alt.Chart(results["product_summary"]).mark_rect(
    stroke=SURFACE, strokeWidth=2
).encode(
    x=alt.X(
        "stars_bin:O",
        title="Average stars (binned to 0.1)",
        axis=alt.Axis(labelAngle=0, labelExpr=SPARSE_DECIMAL_LABELS),
    ),
    # reversed so the biggest products sit on top
    y=alt.Y("size_bucket:O", sort=SIZE_BUCKETS[::-1], title="Reviews per product"),
    color=alt.Color(
        "product_count:Q",
        scale=alt.Scale(type="log", range=SEQ_RAMP),
        title="Products (log)",
    ),
    tooltip=[
        alt.Tooltip("size_bucket:O", title="Reviews per product"),
        alt.Tooltip("stars_bin:O", title="Avg stars bin"),
        alt.Tooltip("product_count:Q", title="Products", format=","),
        alt.Tooltip("total_reviews:Q", title="Total reviews", format=","),
    ],
).properties(
    width=620,
    height=160,
    title=alt.Title(
        "The product landscape: rating vs. popularity",
        subtitle="Each cell counts products at that (avg stars, review volume) combination.",
    ),
)

alt.Chart(...)

## 9. most_helpful_reviews — emphasis scatter with a highlight toggle

Per-review grain survives here (4,315 points, under the cap). The first cut
colored all five star levels with a diverging scale, but no fixed coloring can
serve both questions this chart answers — "where are the critical reviews?"
and "where are the positive ones?". So the emphasis side is a *parameter*: a
radio control (rendered below the chart) switches which group is highlighted.
The two layers' filters and the highlight color are Vega expressions that
reference the `highlight` param, so toggling re-filters client-side with no
recompute. 3★ reviews stay in the gray context in both modes.


In [10]:
# a variable param bound to a radio widget; name= matters, the Vega
# expressions below refer to it as a signal
highlight = alt.param(
    name="highlight",
    value="critical",
    bind=alt.binding_radio(
        options=["critical", "positive"],
        labels=["1-2★ (critical)", "4-5★ (positive)"],
        name="Highlight: ",
    ),
)

HIT = (
    "(highlight === 'critical' && datum.overall <= 2) || "
    "(highlight === 'positive' && datum.overall >= 4)"
)

helpful_base = alt.Chart(results["most_helpful_reviews"]).encode(
    x=alt.X(
        "helpful_votes:Q",
        # nice=False keeps the domain at the data extent (5 to ~2,000) instead
        # of rounding out to full powers of ten
        scale=alt.Scale(type="log", nice=False),
        title="Helpful votes (log)",
        axis=alt.Axis(values=[5, 10, 20, 50, 100, 200, 500, 1000, 2000], format="~s"),
    ),
    y=alt.Y(
        "vote_ratio:Q",
        scale=alt.Scale(domain=[0, 1]),
        title="Helpful / total votes",
        axis=alt.Axis(format="%"),
    ),
    tooltip=[
        alt.Tooltip("title:N", title="Product"),
        alt.Tooltip("reviewer_name:N", title="Reviewer"),
        alt.Tooltip("overall:Q", title="Stars"),
        alt.Tooltip("helpful_votes:Q", title="Helpful votes", format=","),
        alt.Tooltip("total_votes:Q", title="Total votes", format=","),
        alt.Tooltip("vote_ratio:Q", title="Ratio", format=".0%"),
    ],
)

helpful_context = helpful_base.transform_filter(f"!({HIT})").mark_circle(
    size=14, opacity=0.35, color=DEEMPHASIS
).add_params(highlight)

# layered second = always drawn on top; color flips with the toggle so each
# group keeps its diverging-pole hue (red = critical, blue = positive)
helpful_hot = helpful_base.transform_filter(HIT).mark_circle(
    size=34,
    opacity=0.65,
    color=alt.expr(
        f"highlight === 'critical' ? '{DIVERGING_5[0]}' : '{DIVERGING_5[-1]}'"
    ),
)

(helpful_context + helpful_hot).properties(
    width=620,
    height=300,
    title=alt.Title(
        "Each product's most helpful review",
        subtitle="One point per product (top review has ≥5 helpful votes). Toggle below to switch the highlighted group.",
    ),
)


alt.LayerChart(...)

## 10. Scratch

Everything below here is disposable.
